In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline

In [3]:
df_AID_base = pd.read_excel("datasets/Airport_IATA_delays_airline_reported.xlsx")
df_AT = pd.read_excel("datasets/AirportTraffic.xlsx")

In [4]:
df_top20 = (
    df_AT.groupby("APT_ICAO")[["FLT_TOT_1", "FLT_DEP_1", "FLT_ARR_1"]] #Group by airport code
    .sum().sort_values(by="FLT_TOT_1",ascending=False) #Sum the values for each code of the 3 columns indicated
    .head(20).reset_index()) #Change "20" to change the number of airports analysed

# Adding airport's city name and state from original dataset
df_top20 = (df_top20.merge(df_AT[["APT_ICAO", "APT_NAME", "STATE_NAME"]]
                           .drop_duplicates(), on="APT_ICAO", how="left"))
airports_code_list = df_top20["APT_ICAO"].tolist()

In [5]:

df_AID = df_AID_base[df_AID_base["APT_ICAO"].isin(airports_code_list)].dropna()
#Calculating Delay Ratio
df_AID["Delay_Ratio"] = (df_AID["TF"] / df_AID["Total_Flights_Period"]) * 100
#Isolating "adm" predictor
df_AID = (df_AID.groupby(["APT_ICAO", "Year_Lobt", "Month_Lobt"])
          .agg({
            "TF": "sum",
            "Total_Flights_Period": "sum",
            "adm": "sum",
            "Delay_Ratio": "mean" 
        }).reset_index())

In [6]:
consistency_ratio = df_AID.groupby(["APT_ICAO", "Year_Lobt"])["TF"].agg(["mean", "std"]).reset_index()
consistency_ratio["CR"] = consistency_ratio["std"]/consistency_ratio["mean"]

In [7]:
df = df_AID.merge(consistency_ratio[["APT_ICAO", "Year_Lobt", "CR"]], on=["APT_ICAO", "Year_Lobt"], how="left")
df.head()

,APT_ICAO,Year_Lobt,Month_Lobt,TF,Total_Flights_Period,adm,Delay_Ratio,CR
0,EDDF,2023,April,16336,211616,12.099695,7.719643,0.198526
1,EDDF,2023,August,21990,240295,20.627308,9.151252,0.198526
2,EDDF,2023,December,17029,185113,18.473322,9.199246,0.198526
3,EDDF,2023,February,11685,156502,15.133283,7.466358,0.198526
4,EDDF,2023,January,12250,173978,12.483877,7.041120,0.198526


In [8]:
df["Delay_Prone"] = np.where(df["Delay_Ratio"] >= 7, 1, 0)
df.sort_values("Delay_Prone", ascending=True)

,APT_ICAO,Year_Lobt,Month_Lobt,TF,Total_Flights_Period,adm,Delay_Ratio,CR,Delay_Prone
239,LEMD,2024,September,5552,147866,9.309152,3.754751,0.182060,0
324,LIRF,2024,April,3670,88434,8.453095,4.149988,0.389915,0
323,LIRF,2023,September,7236,157386,11.132426,4.597614,0.332158,0
322,LIRF,2023,October,7632,162554,10.477933,4.695055,0.332158,0
321,LIRF,2023,November,4967,120037,8.721286,4.137891,0.332158,0
...,...,...,...,...,...,...,...,...,...
41,EDDM,2024,July,15833,179962,19.529001,8.797968,0.293847,1
19,EDDF,2024,March,15090,195925,13.059696,7.701927,0.217284,1
18,EDDF,2024,June,19572,219232,17.353986,8.927529,0.217284,1
25,EDDM,2023,August,12640,172465,17.367965,7.329023,0.250041,1


In [9]:
months = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}
df["Month_Lobt"] = df["Month_Lobt"].map(months)


# Prediction

In [ ]:
features = ["Total_Flights_Period", "adm", "CR"] # Model features
X = df[features] # predictors
y = df["Delay_Prone"] # predicted variable

numeric_features = ["Total_Flights_Period","adm", "CR"] # to be scaled

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features), # features to be scaled
    ]
)

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor), # Actual scaling
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=1, n_estimators=200)) # Model used, we use balanced weight to moderate class split 
])

#To avoid the model memorizing patterns rather than predict them, we train on 80% of the airports and test on the remaining 20%
unique_airports = df["APT_ICAO"].unique()
# For example, train on first 14 airports, test on remaining 6
train_airports = unique_airports[:16]
test_airports = unique_airports[16:]

X_train = df[df["APT_ICAO"].isin(train_airports)][features]
y_train = df[df["APT_ICAO"].isin(train_airports)]["Delay_Prone"]

X_test = df[df["APT_ICAO"].isin(test_airports)][features]
y_test = df[df["APT_ICAO"].isin(test_airports)]["Delay_Prone"]

# Fit and evaluate
rf_model.fit(X_train, y_train)# Model training

,steps,"[('preprocess', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [51]:
y_pred = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred)) # (TP + TN)/(FP + FN)
print(confusion_matrix(y_test, y_pred)) # [TP, FP], [FN, TN]
print(classification_report(y_test, y_pred)) 

Accuracy: 0.968421052631579
[[92  1]
 [ 2  0]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.98        93
           1       0.00      0.00      0.00         2

    accuracy                           0.97        95
   macro avg       0.49      0.49      0.49        95
weighted avg       0.96      0.97      0.96        95



In [52]:
rf_model2 = Pipeline(steps=[
    ("preprocess", preprocessor), # Actual scaling
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=1, n_estimators=200)) # Model used, we use balanced weight to moderate class split 
])

In [53]:
acc = []
unique_airports = df["APT_ICAO"].unique()
# For example, train on first 14 airports, test on remaining 6
train_airports = unique_airports[:16]
test_airports = unique_airports[16:]

X_train = df[df["APT_ICAO"].isin(train_airports)][features]
y_train = df[df["APT_ICAO"].isin(train_airports)]["Delay_Prone"]

X_test = df[df["APT_ICAO"].isin(test_airports)][features]
y_test = df[df["APT_ICAO"].isin(test_airports)]["Delay_Prone"]

for i in range(100):
    rf_model2.fit(X_train, y_train)# Model training
    y_pred = rf_model2.predict(X_test)
    acc.append(accuracy_score(y_test, y_pred))
print(np.mean(acc))

0.9684210526315791


In [30]:
importances = rf_model.named_steps["classifier"].feature_importances_
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nFeature Importances:")
print(importance_df)


Feature Importances:
                Feature  Importance
1  Total_Flights_Period    0.451317
2                    CR    0.445560
0            Month_Lobt    0.103123


In [43]:
df[["Total_Flights_Period", "adm", "CR", "Delay_Prone"]].corr()

,Total_Flights_Period,adm,CR,Delay_Prone
Total_Flights_Period,1.000000,0.364938,-0.480967,0.226168
adm,0.364938,1.000000,-0.122351,0.336029
CR,-0.480967,-0.122351,1.000000,-0.144006
Delay_Prone,0.226168,0.336029,-0.144006,1.000000
